In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
import math
import plotly.graph_objects as go

In [3]:
def associated_legendre(l,m,x):
    if isinstance(l, bool) or not isinstance(l, (int, np.integer)):
        raise TypeError("l must be an integer.")
    if isinstance(m, bool) or not isinstance(m, (int, np.integer)):
        raise TypeError("m must be an integer.")
    if l < 0:
        raise ValueError("l must be non-negative.")
    if m < 0 or m > l:
        raise ValueError("m must satisfy 0 <= m <= l.")

    x_arr = np.asarray(x, dtype=float)

    if not np.all(np.isfinite(x_arr)):
        raise ValueError("All values of x must be finite.")

    if np.any(x_arr < -1) or np.any(x_arr > 1):
        raise ValueError("All values of x must lie in the interval [-1, 1].")

    sqrt_term = np.sqrt(np.clip(1.0 - x_arr**2, 0.0, None))

    p_mm = np.ones_like(x_arr, dtype=float)

    for order in range(1, m + 1):
        p_mm = -(2 * order - 1) * sqrt_term * p_mm

    if l == m:
        return p_mm

    p_mplus1_m = (2 * m + 1) * x_arr * p_mm
    if l == m + 1:
        return p_mplus1_m

    p_previous_previous = p_mm
    p_previous = p_mplus1_m

    for degree in range(m + 2, l + 1):

        p_current = (
            (2 * degree - 1) * x_arr * p_previous
            - (degree + m - 1) * p_previous_previous
        ) / (degree - m)

        p_previous_previous = p_previous
        p_previous = p_current

    return p_previous

In [4]:
def spherical_harmonic(l, m, theta, phi):
    if isinstance(l, bool) or not isinstance(l, (int, np.integer)):
        raise TypeError("l must be an integer.")
    if isinstance(m, bool) or not isinstance(m, (int, np.integer)):
        raise TypeError("m must be an integer.")
    if l < 0:
        raise ValueError("l must be non-negative.")
    if m < -l or m > l:
        raise ValueError("m must satisfy -l <= m <= l.")

    theta_arr = np.asarray(theta, dtype=float)
    phi_arr   = np.asarray(phi, dtype=float)

    m_abs = abs(m)

    x = np.cos(theta_arr)
    legendre_part = associated_legendre(l,m_abs,x)

    normalization_numerator = ((2*l + 1)*math.factorial(l - m_abs))
    normalization_denominator = (4*np.pi*math.factorial(l + m_abs))
    normalization = np.sqrt(normalization_numerator/normalization_denominator)

    phase_term = np.exp(1j*m_abs*phi_arr)
    y_positive = (normalization*legendre_part*phase_term)

    if m >= 0:
        return y_positive

    return (-1)**m_abs * np.conjugate(y_positive)